# Phase 3: Confluence Scoring Validation

**Goal:** Match Pine script's confluence scoring exactly

## Scoring Components (max 5 points)
1. **ORB Break** (0/1) — breakout condition met
2. **SSL Hybrid** (0/1) — close vs HMA baseline + gap widening
3. **WAE** (0/1) — trend > explosion + acceleration
4. **QQE** (0/1) — dual RSI + Bollinger + consecutive increasing
5. **Volume** (0/1) — volRatio thresholds then rounded

## Known Issue
Earlier testing showed Python was firing when Pine didn't, suggesting volume scoring was too generous.

In [3]:
# Cell 1: Setup and load data
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')

import pandas as pd
import numpy as np
from dotenv import load_dotenv
load_dotenv(r'C:\Users\phemm\orb_lab\.env')

from data_collector import PolygonDataCollector

# Fetch with enough history
collector = PolygonDataCollector()
df = collector.fetch_bars('NVDA', days_back=120, bar_size=1)
print(f"✓ Loaded {len(df)} bars")
print(f"Date range: {df.index[0].date()} to {df.index[-1].date()}")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching NVDA (1-min bars, 120 days)...
  ✓ Loading from cache (0.2 hours old)
✓ Loaded 31704 bars
Date range: 2025-09-02 to 2025-12-26


In [4]:
# Cell 2: Helper functions for moving averages (matching Pine)

def calc_sma(series, period):
    """Simple Moving Average"""
    return series.rolling(period, min_periods=1).mean()

def calc_ema(series, period):
    """Exponential Moving Average - Pine uses alpha = 2/(period+1)"""
    return series.ewm(span=period, adjust=False).mean()

def calc_rma(series, period):
    """Wilder's Moving Average (RMA) - Pine uses alpha = 1/period"""
    return series.ewm(alpha=1/period, adjust=False).mean()

def calc_hma(series, period):
    """Hull Moving Average"""
    half_period = int(period / 2)
    sqrt_period = int(np.sqrt(period))
    
    wma_half = series.rolling(half_period, min_periods=1).apply(
        lambda x: np.sum(np.arange(1, len(x)+1) * x) / np.sum(np.arange(1, len(x)+1)), raw=True)
    wma_full = series.rolling(period, min_periods=1).apply(
        lambda x: np.sum(np.arange(1, len(x)+1) * x) / np.sum(np.arange(1, len(x)+1)), raw=True)
    
    raw = 2 * wma_half - wma_full
    hma = raw.rolling(sqrt_period, min_periods=1).apply(
        lambda x: np.sum(np.arange(1, len(x)+1) * x) / np.sum(np.arange(1, len(x)+1)), raw=True)
    
    return hma

def calc_jma(series, period, phase=0, power=2):
    """Jurik Moving Average (simplified approximation)"""
    # JMA is proprietary, this is an approximation
    beta = 0.45 * (period - 1) / (0.45 * (period - 1) + 2)
    alpha = beta ** power
    
    jma = series.copy()
    e0 = series.copy()
    e1 = series.copy()
    e2 = series.copy()
    
    for i in range(1, len(series)):
        e0.iloc[i] = (1 - alpha) * series.iloc[i] + alpha * e0.iloc[i-1]
        e1.iloc[i] = (series.iloc[i] - e0.iloc[i]) * (1 - beta) + beta * e1.iloc[i-1]
        e2.iloc[i] = e0.iloc[i] + e1.iloc[i]
        jma.iloc[i] = e2.iloc[i]
    
    return jma

def ma(series, period, ma_type='JMA'):
    """Generic MA selector matching Pine's ma() function"""
    if ma_type == 'SMA':
        return calc_sma(series, period)
    elif ma_type == 'EMA':
        return calc_ema(series, period)
    elif ma_type == 'HMA':
        return calc_hma(series, period)
    elif ma_type == 'JMA':
        return calc_jma(series, period)
    else:
        return calc_sma(series, period)

print("✓ Moving average functions defined")

✓ Moving average functions defined


In [5]:
# Cell 3: SSL Hybrid Calculation

def calc_ssl_hybrid(df, baseline_length=20, ssl_length=10, ssl_type='JMA', 
                    use_momentum_filter=True):
    """
    Pine SSL Logic:
        sslBaseline = ta.hma(close, sslBaselineLengthEff)  # 20 for NVDA
        sslUp = ma(high, sslLengthEff, sslType)           # JMA(10)
        sslDown = ma(low, sslLengthEff, sslType)          # JMA(10)
        
        sslBullConfirmation = close > sslBaseline
        sslBearConfirmation = close < sslBaseline
        
        # Gap widening (acceleration)
        sslGapBull = close - sslBaseline
        sslAccelBull = sslGapBull > sslGapBull[1]
        
        # Score (with momentum filter)
        sslScoreBull = sslBullConfirmation AND sslAccelBull ? 1 : 0
    """
    df = df.copy()
    
    # SSL Baseline (HMA on close)
    df['ssl_baseline'] = calc_hma(df['close'], baseline_length)
    
    # SSL Up/Down channels
    df['ssl_up'] = ma(df['high'], ssl_length, ssl_type)
    df['ssl_down'] = ma(df['low'], ssl_length, ssl_type)
    
    # Basic confirmation
    df['ssl_bull_confirm'] = df['close'] > df['ssl_baseline']
    df['ssl_bear_confirm'] = df['close'] < df['ssl_baseline']
    
    # Gap widening (acceleration)
    df['ssl_gap_bull'] = df['close'] - df['ssl_baseline']
    df['ssl_gap_bear'] = df['ssl_baseline'] - df['close']
    
    df['ssl_accel_bull'] = df['ssl_gap_bull'] > df['ssl_gap_bull'].shift(1)
    df['ssl_accel_bear'] = df['ssl_gap_bear'] > df['ssl_gap_bear'].shift(1)
    
    # SSL Score
    if use_momentum_filter:
        df['ssl_score_bull'] = ((df['ssl_bull_confirm']) & (df['ssl_accel_bull'])).astype(int)
        df['ssl_score_bear'] = ((df['ssl_bear_confirm']) & (df['ssl_accel_bear'])).astype(int)
    else:
        df['ssl_score_bull'] = df['ssl_bull_confirm'].astype(int)
        df['ssl_score_bear'] = df['ssl_bear_confirm'].astype(int)
    
    return df

# Apply SSL with NVDA presets
df = calc_ssl_hybrid(df, baseline_length=20, ssl_length=10, ssl_type='JMA')
print("✓ SSL Hybrid calculated")

✓ SSL Hybrid calculated


In [6]:
# Cell 4: WAE (Waddah Attar Explosion) Calculation

def calc_wae(df, fast_ema=20, slow_ema=40, bb_length=20, bb_mult=2.0, 
             sensitivity=175, use_acceleration=True):
    """
    Pine WAE Logic:
        fastMA = ta.ema(close, 20)
        slowMA = ta.ema(close, 40)
        macd = fastMA - slowMA
        t1 = (macd - macd[1]) * sensitivity
        
        basis = ta.sma(close, 20)
        dev = bbMult * ta.stdev(close, 20)
        explosionLine = (basis + dev) - (basis - dev) = 2 * dev
        
        trendUp = t1 >= 0 ? t1 : 0
        trendDown = t1 < 0 ? -t1 : 0
        
        waeBullConfirmation = trendUp > explosionLine
        waeAccelBull = trendUp > trendUp[1] AND trendUp > explosionLine
    """
    df = df.copy()
    
    # MACD-style calculation
    df['wae_fast_ma'] = calc_ema(df['close'], fast_ema)
    df['wae_slow_ma'] = calc_ema(df['close'], slow_ema)
    df['wae_macd'] = df['wae_fast_ma'] - df['wae_slow_ma']
    
    # Trend line (sensitivity applied to MACD change)
    df['wae_t1'] = (df['wae_macd'] - df['wae_macd'].shift(1)) * sensitivity
    
    # Explosion line (Bollinger Band width)
    df['wae_basis'] = calc_sma(df['close'], bb_length)
    df['wae_dev'] = df['close'].rolling(bb_length).std() * bb_mult
    df['wae_explosion'] = 2 * df['wae_dev']  # Upper - Lower = 2 * dev
    
    # Trend Up/Down
    df['wae_trend_up'] = df['wae_t1'].clip(lower=0)
    df['wae_trend_down'] = (-df['wae_t1']).clip(lower=0)
    
    # Basic confirmation
    df['wae_bull_confirm'] = df['wae_trend_up'] > df['wae_explosion']
    df['wae_bear_confirm'] = df['wae_trend_down'] > df['wae_explosion']
    
    # Acceleration
    df['wae_accel_bull'] = (df['wae_trend_up'] > df['wae_trend_up'].shift(1)) & df['wae_bull_confirm']
    df['wae_accel_bear'] = (df['wae_trend_down'] > df['wae_trend_down'].shift(1)) & df['wae_bear_confirm']
    
    # WAE Score
    if use_acceleration:
        df['wae_score_bull'] = (df['wae_bull_confirm'] & df['wae_accel_bull']).astype(int)
        df['wae_score_bear'] = (df['wae_bear_confirm'] & df['wae_accel_bear']).astype(int)
    else:
        df['wae_score_bull'] = df['wae_bull_confirm'].astype(int)
        df['wae_score_bear'] = df['wae_bear_confirm'].astype(int)
    
    return df

# Apply WAE with NVDA preset (sensitivity=175)
df = calc_wae(df, sensitivity=175)
print("✓ WAE calculated")

✓ WAE calculated


In [7]:
# Cell 5: QQE Calculation

def calc_rsi(series, period):
    """RSI calculation matching Pine"""
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = (-delta).clip(lower=0)
    
    avg_gain = calc_rma(gain, period)
    avg_loss = calc_rma(loss, period)
    
    rs = avg_gain / avg_loss.replace(0, np.nan)
    rsi = 100 - (100 / (1 + rs))
    return rsi.fillna(50)

def calc_qqe_line(df, rsi_length=14, smoothing=5, qqe_factor=3.0):
    """
    Pine QQE calculation:
        wildersLength = rsiLength * 2 - 1
        rsi = ta.rsi(source, rsiLength)
        rsiMA = ta.ema(rsi, smoothingFactor)
        atrRsi = math.abs(rsiMA - rsiMA[1])
        smoothedAtrRsi = ta.ema(ta.ema(atrRsi, wildersLength), wildersLength)
        dar = smoothedAtrRsi * qqeFactor
        
        Then trailing stop logic for QQE line
    """
    wilders_length = rsi_length * 2 - 1
    
    # RSI and smoothed RSI
    rsi = calc_rsi(df['close'], rsi_length)
    rsi_ma = calc_ema(rsi, smoothing)
    
    # ATR of RSI
    atr_rsi = abs(rsi_ma - rsi_ma.shift(1))
    smoothed_atr = calc_ema(calc_ema(atr_rsi, wilders_length), wilders_length)
    dar = smoothed_atr * qqe_factor
    
    # QQE Line (trailing stop on RSI)
    qqe_line = pd.Series(index=df.index, dtype=float)
    trend = pd.Series(index=df.index, dtype=int)
    
    qqe_line.iloc[0] = rsi_ma.iloc[0]
    trend.iloc[0] = 1
    
    for i in range(1, len(df)):
        prev_qqe = qqe_line.iloc[i-1]
        curr_rsi = rsi_ma.iloc[i]
        curr_dar = dar.iloc[i] if not np.isnan(dar.iloc[i]) else 0
        
        new_short = curr_rsi + curr_dar
        new_long = curr_rsi - curr_dar
        
        if np.isnan(prev_qqe):
            qqe_line.iloc[i] = curr_rsi
            trend.iloc[i] = 1
            continue
        
        if rsi_ma.iloc[i-1] > prev_qqe and curr_rsi > prev_qqe:
            qqe_line.iloc[i] = max(prev_qqe, new_long)
            trend.iloc[i] = 1
        elif rsi_ma.iloc[i-1] < prev_qqe and curr_rsi < prev_qqe:
            qqe_line.iloc[i] = min(prev_qqe, new_short)
            trend.iloc[i] = -1
        elif curr_rsi > prev_qqe:
            qqe_line.iloc[i] = new_long
            trend.iloc[i] = 1
        else:
            qqe_line.iloc[i] = new_short
            trend.iloc[i] = -1
    
    return qqe_line, rsi_ma, trend


def calc_qqe_score(df, rsi1_length=14, rsi1_smooth=5, qqe_factor1=3.0,
                   rsi2_length=6, rsi2_smooth=3, qqe_factor2=1.61,
                   bb_length=50, bb_mult=0.35, threshold=10,
                   consecutive_bars=3, use_momentum_filter=True):
    """
    Pine QQE Scoring:
        Primary QQE: RSI(14), smooth(5), factor(3.0)
        Secondary QQE: RSI(6), smooth(3), factor(1.61)
        
        bollingerBasis = sma(primaryQQE - 50, 50)
        bollingerUpper = bollingerBasis + 0.35 * stdev
        
        rawQQEBullCondition = secondaryRSI - 50 > threshold AND primaryRSI - 50 > bollingerUpper
        
        Then consecutive increasing check
    """
    df = df.copy()
    
    # Primary QQE
    primary_qqe, primary_rsi, primary_trend = calc_qqe_line(df, rsi1_length, rsi1_smooth, qqe_factor1)
    df['qqe_primary_line'] = primary_qqe
    df['qqe_primary_rsi'] = primary_rsi
    
    # Secondary QQE
    secondary_qqe, secondary_rsi, secondary_trend = calc_qqe_line(df, rsi2_length, rsi2_smooth, qqe_factor2)
    df['qqe_secondary_rsi'] = secondary_rsi
    
    # Bollinger on primary QQE
    qqe_centered = df['qqe_primary_line'] - 50
    df['qqe_bb_basis'] = calc_sma(qqe_centered, bb_length)
    df['qqe_bb_dev'] = qqe_centered.rolling(bb_length).std() * bb_mult
    df['qqe_bb_upper'] = df['qqe_bb_basis'] + df['qqe_bb_dev']
    df['qqe_bb_lower'] = df['qqe_bb_basis'] - df['qqe_bb_dev']
    
    # Raw conditions
    primary_centered = df['qqe_primary_rsi'] - 50
    secondary_centered = df['qqe_secondary_rsi'] - 50
    
    df['qqe_raw_bull'] = (secondary_centered > threshold) & (primary_centered > df['qqe_bb_upper'])
    df['qqe_raw_bear'] = (secondary_centered < -threshold) & (primary_centered < df['qqe_bb_lower'])
    
    # Signal strength
    df['bull_strength'] = np.minimum(secondary_centered - threshold, primary_centered - df['qqe_bb_upper'])
    df['bear_strength'] = np.minimum(-threshold - secondary_centered, df['qqe_bb_lower'] - primary_centered)
    
    # Consecutive increasing counter
    df['bull_increasing'] = 0
    df['bear_increasing'] = 0
    
    bull_count = 0
    bear_count = 0
    prev_bull_strength = 0
    prev_bear_strength = 0
    
    for i in range(len(df)):
        # Bull
        if df['qqe_raw_bull'].iloc[i]:
            if df['bull_strength'].iloc[i] > prev_bull_strength:
                bull_count += 1
            else:
                bull_count = 0
            prev_bull_strength = df['bull_strength'].iloc[i]
        else:
            bull_count = 0
            prev_bull_strength = 0
        df.iloc[i, df.columns.get_loc('bull_increasing')] = bull_count
        
        # Bear
        if df['qqe_raw_bear'].iloc[i]:
            if df['bear_strength'].iloc[i] > prev_bear_strength:
                bear_count += 1
            else:
                bear_count = 0
            prev_bear_strength = df['bear_strength'].iloc[i]
        else:
            bear_count = 0
            prev_bear_strength = 0
        df.iloc[i, df.columns.get_loc('bear_increasing')] = bear_count
    
    # Final QQE signal (consecutive bars requirement)
    df['qqe_bull_signal'] = (df['bull_increasing'] >= consecutive_bars - 1) & df['qqe_raw_bull']
    df['qqe_bear_signal'] = (df['bear_increasing'] >= consecutive_bars - 1) & df['qqe_raw_bear']
    
    # Momentum direction filter
    df['qqe_momentum_rising'] = df['qqe_primary_rsi'] > df['qqe_primary_rsi'].shift(1)
    df['qqe_momentum_falling'] = df['qqe_primary_rsi'] < df['qqe_primary_rsi'].shift(1)
    
    # QQE Score
    if use_momentum_filter:
        df['qqe_score_bull'] = (df['qqe_bull_signal'] & df['qqe_momentum_rising']).astype(int)
        df['qqe_score_bear'] = (df['qqe_bear_signal'] & df['qqe_momentum_falling']).astype(int)
    else:
        df['qqe_score_bull'] = df['qqe_bull_signal'].astype(int)
        df['qqe_score_bear'] = df['qqe_bear_signal'].astype(int)
    
    return df

# Apply QQE
df = calc_qqe_score(df)
print("✓ QQE calculated")

✓ QQE calculated


In [8]:
# Cell 6: Volume Score Calculation

def calc_volume_score(df, vol_lookback=20):
    """
    Pine Volume Logic:
        volAvg = ta.sma(volume, volLookback)
        volRatio = volume / volAvg
        
        volScore = 0
        if volRatio >= 1.5: volScore = 1.0
        else if volRatio >= 1.0: volScore = 0.5
        else if volRatio >= 0.5: volScore = 0.25
        
        # For confluence total: int(math.round(volScore))
        # 1.0 → 1, 0.5 → 1 (rounds up!), 0.25 → 0
    """
    df = df.copy()
    
    df['vol_avg'] = calc_sma(df['volume'], vol_lookback)
    df['vol_ratio'] = df['volume'] / df['vol_avg'].replace(0, np.nan)
    
    # Raw volume score
    conditions = [
        df['vol_ratio'] >= 1.5,
        df['vol_ratio'] >= 1.0,
        df['vol_ratio'] >= 0.5
    ]
    choices = [1.0, 0.5, 0.25]
    df['vol_score_raw'] = np.select(conditions, choices, default=0.0)
    
    # Rounded for confluence total (Pine uses int(math.round(volScore)))
    # Note: math.round(0.5) = 1 in Pine (rounds to nearest even, but 0.5 → 1)
    df['vol_score'] = df['vol_score_raw'].round().astype(int)
    
    return df

# Apply volume score
df = calc_volume_score(df)
print("✓ Volume score calculated")

# Show distribution
print(f"\nVolume score distribution:")
print(df['vol_score'].value_counts().sort_index())

✓ Volume score calculated

Volume score distribution:
vol_score
0    28790
1     2914
Name: count, dtype: int64


In [9]:
# Cell 7: Test on Oct 29 (the trade that fired with confluence 4)

test_date = '2025-10-29'
test_time = '09:59:00-04:00'  # Approximate breakout time

# Find bars around breakout
day_df = df[df.index.date.astype(str) == test_date].copy()

# Find first bar after 09:35 where close > ORB high
# (We need to add ORB detection first)
def detect_orb(df, orb_minutes=5):
    def is_in_orb(ts):
        bar_total = ts.hour * 60 + ts.minute
        return 9*60+30 <= bar_total < 9*60+30+orb_minutes
    
    df = df.copy()
    df['in_session'] = df.index.map(is_in_orb)
    df['is_new_session'] = df['in_session'] & ~df['in_session'].shift(1, fill_value=False)
    
    df['orb_high'] = np.nan
    df['orb_low'] = np.nan
    current_high, current_low = np.nan, np.nan
    
    for i in range(len(df)):
        if df['is_new_session'].iloc[i]:
            current_high = df['high'].iloc[i]
            current_low = df['low'].iloc[i]
        elif df['in_session'].iloc[i]:
            current_high = max(current_high, df['high'].iloc[i])
            current_low = min(current_low, df['low'].iloc[i])
        df.iloc[i, df.columns.get_loc('orb_high')] = current_high
        df.iloc[i, df.columns.get_loc('orb_low')] = current_low
    
    df['orb_complete'] = ~df['in_session'] & df['orb_high'].notna()
    return df

df = detect_orb(df)

# Now find the breakout bar on Oct 29
day_df = df[df.index.date.astype(str) == test_date].copy()
orb_high = day_df['orb_high'].iloc[-1]
orb_low = day_df['orb_low'].iloc[-1]

print(f"=== Confluence Test on {test_date} ===")
print(f"ORB High: {orb_high:.2f}")
print(f"ORB Low: {orb_low:.2f}")

# Find first LONG breakout
for ts, row in day_df.iterrows():
    if not row['orb_complete']:
        continue
    if row['close'] > orb_high:
        print(f"\n📍 LONG breakout at {ts.strftime('%H:%M')}")
        print(f"   Close: {row['close']:.2f}")
        print(f"\n   === Confluence Scores ===")
        print(f"   ORB Break:  1 (by definition)")
        print(f"   SSL Score:  {row['ssl_score_bull']}")
        print(f"   WAE Score:  {row['wae_score_bull']}")
        print(f"   QQE Score:  {row['qqe_score_bull']}")
        print(f"   Vol Score:  {row['vol_score']} (raw: {row['vol_score_raw']:.2f}, ratio: {row['vol_ratio']:.2f})")
        
        total = 1 + row['ssl_score_bull'] + row['wae_score_bull'] + row['qqe_score_bull'] + row['vol_score']
        print(f"\n   TOTAL: {total}/5")
        print(f"\n   Pine showed: 5/5 with volume, 4/5 without")
        break

=== Confluence Test on 2025-10-29 ===
ORB High: 211.08
ORB Low: 207.09

📍 LONG breakout at 09:36
   Close: 211.32

   === Confluence Scores ===
   ORB Break:  1 (by definition)
   SSL Score:  0
   WAE Score:  0
   QQE Score:  1
   Vol Score:  0 (raw: 0.50, ratio: 1.39)

   TOTAL: 2/5

   Pine showed: 5/5 with volume, 4/5 without


In [10]:
# Cell 8: Test on Oct 30 (the SKIP trade - check confluence)

test_date = '2025-10-30'
day_df = df[df.index.date.astype(str) == test_date].copy()

orb_high = day_df['orb_high'].iloc[-1]
orb_low = day_df['orb_low'].iloc[-1]

print(f"=== Confluence Test on {test_date} (SKIP trade) ===")
print(f"ORB High: {orb_high:.2f}")
print(f"ORB Low: {orb_low:.2f}")

# Find first SHORT breakout
for ts, row in day_df.iterrows():
    if not row['orb_complete']:
        continue
    if row['close'] < orb_low:
        print(f"\n📍 SHORT breakout at {ts.strftime('%H:%M')}")
        print(f"   Close: {row['close']:.2f}")
        print(f"\n   === Confluence Scores ===")
        print(f"   ORB Break:  1 (by definition)")
        print(f"   SSL Score:  {row['ssl_score_bear']}")
        print(f"   WAE Score:  {row['wae_score_bear']}")
        print(f"   QQE Score:  {row['qqe_score_bear']}")
        print(f"   Vol Score:  {row['vol_score']} (raw: {row['vol_score_raw']:.2f}, ratio: {row['vol_ratio']:.2f})")
        
        total = 1 + row['ssl_score_bear'] + row['wae_score_bear'] + row['qqe_score_bear'] + row['vol_score']
        print(f"\n   TOTAL: {total}/5")
        
        # Show what Pine had (from screenshot)
        print(f"\n   === Pine showed (from screenshot) ===")
        print(f"   ORB Break: ✓")
        print(f"   SSL: ✓")
        print(f"   WAE: ✓")
        print(f"   QQE: ✓")
        print(f"   Volume: Avg")
        print(f"   Score: 5/5")
        break

=== Confluence Test on 2025-10-30 (SKIP trade) ===
ORB High: 206.16
ORB Low: 202.87

📍 SHORT breakout at 09:35
   Close: 202.35

   === Confluence Scores ===
   ORB Break:  1 (by definition)
   SSL Score:  0
   WAE Score:  1
   QQE Score:  1
   Vol Score:  1 (raw: 1.00, ratio: 1.59)

   TOTAL: 4/5

   === Pine showed (from screenshot) ===
   ORB Break: ✓
   SSL: ✓
   WAE: ✓
   QQE: ✓
   Volume: Avg
   Score: 5/5


In [11]:
# Cell 9: Debug individual components at breakout bar

# Use the Oct 30 SHORT breakout bar
ts = '2025-10-30 09:35:00-04:00'
row = df.loc[ts]

print("=== Component Debug for Oct 30 09:35 ===")

print("\n--- SSL ---")
print(f"SSL Baseline: {row['ssl_baseline']:.2f}")
print(f"Close: {row['close']:.2f}")
print(f"Bear Confirm (close < baseline): {row['ssl_bear_confirm']}")
print(f"Gap Bear: {row['ssl_gap_bear']:.4f}")
gap_prev = df.loc[df.index[df.index.get_loc(ts)-1], 'ssl_gap_bear']
print(f"Gap Bear Prev: {gap_prev:.4f}")
print(f"Accel Bear (gap widening): {row['ssl_accel_bear']}")
print(f"SSL Score Bear: {row['ssl_score_bear']}")

print("\n--- WAE ---")
print(f"Trend Down: {row['wae_trend_down']:.2f}")
print(f"Explosion Line: {row['wae_explosion']:.2f}")
print(f"Bear Confirm (trend > explosion): {row['wae_bear_confirm']}")
trend_prev = df.loc[df.index[df.index.get_loc(ts)-1], 'wae_trend_down']
print(f"Trend Down Prev: {trend_prev:.2f}")
print(f"Accel Bear: {row['wae_accel_bear']}")
print(f"WAE Score Bear: {row['wae_score_bear']}")

print("\n--- QQE ---")
print(f"Primary RSI: {row['qqe_primary_rsi']:.2f}")
print(f"Secondary RSI: {row['qqe_secondary_rsi']:.2f}")
print(f"BB Lower: {row['qqe_bb_lower']:.2f}")
print(f"Raw Bear Condition: {row['qqe_raw_bear']}")
print(f"Bear Increasing Count: {row['bear_increasing']}")
print(f"QQE Bear Signal: {row['qqe_bear_signal']}")
print(f"Momentum Falling: {row['qqe_momentum_falling']}")
print(f"QQE Score Bear: {row['qqe_score_bear']}")

print("\n--- Volume ---")
print(f"Volume: {row['volume']:,.0f}")
print(f"Vol Avg: {row['vol_avg']:,.0f}")
print(f"Vol Ratio: {row['vol_ratio']:.2f}")
print(f"Vol Score Raw: {row['vol_score_raw']:.2f}")
print(f"Vol Score (rounded): {row['vol_score']}")

=== Component Debug for Oct 30 09:35 ===

--- SSL ---
SSL Baseline: 203.79
Close: 202.35
Bear Confirm (close < baseline): True
Gap Bear: 1.4372
Gap Bear Prev: 1.6268
Accel Bear (gap widening): False
SSL Score Bear: 0

--- WAE ---
Trend Down: 23.59
Explosion Line: 6.55
Bear Confirm (trend > explosion): True
Trend Down Prev: 22.62
Accel Bear: True
WAE Score Bear: 1

--- QQE ---
Primary RSI: 22.88
Secondary RSI: 10.63
BB Lower: -0.57
Raw Bear Condition: True
Bear Increasing Count: 6
QQE Bear Signal: True
Momentum Falling: True
QQE Score Bear: 1

--- Volume ---
Volume: 1,857,977
Vol Avg: 1,171,459
Vol Ratio: 1.59
Vol Score Raw: 1.00
Vol Score (rounded): 1


In [12]:
# Debug HMA calculation at 09:35
ts = '2025-10-30 09:35:00-04:00'
ts_prev = '2025-10-30 09:34:00-04:00'

print("=== HMA Baseline Debug ===")
print(f"Python baseline 09:34: {df.loc[ts_prev, 'ssl_baseline']:.2f}")
print(f"Python baseline 09:35: {df.loc[ts, 'ssl_baseline']:.2f}")
print(f"\nPine baseline 09:34: 204.50")
print(f"Pine baseline 09:35: 204.13")
print(f"\nGap: Python is {204.13 - df.loc[ts, 'ssl_baseline']:.2f} lower than Pine")

=== HMA Baseline Debug ===
Python baseline 09:34: 204.54
Python baseline 09:35: 203.79

Pine baseline 09:34: 204.50
Pine baseline 09:35: 204.13

Gap: Python is 0.34 lower than Pine


In [13]:
# Test corrected HMA using pandas built-in
def calc_hma_v2(series, period):
    """Hull Moving Average - corrected version"""
    import numpy as np
    
    half_period = max(1, int(period / 2))
    sqrt_period = max(1, int(np.sqrt(period)))
    
    # WMA helper
    def wma(s, p):
        weights = np.arange(1, p + 1)
        return s.rolling(p).apply(lambda x: np.dot(x, weights[-len(x):]) / weights[-len(x):].sum(), raw=True)
    
    wma_half = wma(series, half_period)
    wma_full = wma(series, period)
    
    raw = 2 * wma_half - wma_full
    hma = wma(raw, sqrt_period)
    
    return hma

# Recalculate SSL baseline with corrected HMA
df['ssl_baseline_v2'] = calc_hma_v2(df['close'], 20)

ts = '2025-10-30 09:35:00-04:00'
ts_prev = '2025-10-30 09:34:00-04:00'

print("=== HMA v2 Test ===")
print(f"v2 baseline 09:34: {df.loc[ts_prev, 'ssl_baseline_v2']:.2f}")
print(f"v2 baseline 09:35: {df.loc[ts, 'ssl_baseline_v2']:.2f}")
print(f"\nPine 09:34: 204.50 | Pine 09:35: 204.13")

=== HMA v2 Test ===
v2 baseline 09:34: 204.54
v2 baseline 09:35: 203.79

Pine 09:34: 204.50 | Pine 09:35: 204.13


In [14]:
# Check if close prices match
ts = '2025-10-30 09:35:00-04:00'
ts_prev = '2025-10-30 09:34:00-04:00'

print("=== Close Price Check ===")
print(f"Python close 09:34: {df.loc[ts_prev, 'close']:.2f}")
print(f"Python close 09:35: {df.loc[ts, 'close']:.2f}")
print(f"\nWhat does TradingView show for these bars?")

=== Close Price Check ===
Python close 09:34: 202.91
Python close 09:35: 202.35

What does TradingView show for these bars?


In [15]:
# Check if pre-market bars are affecting HMA
ts = '2025-10-30 09:35:00-04:00'

# How many bars before 09:35 on Oct 30?
oct30_df = df[df.index.date == pd.Timestamp('2025-10-30').date()]
before_935 = oct30_df[oct30_df.index < pd.Timestamp(ts)]

print(f"Bars on Oct 30 before 09:35: {len(before_935)}")
print(f"First bar: {before_935.index[0]}")
print(f"Last 5 bars before 09:35:")
print(before_935[['close']].tail())

Bars on Oct 30 before 09:35: 5
First bar: 2025-10-30 09:30:00-04:00
Last 5 bars before 09:35:
                              close
timestamp                          
2025-10-30 09:30:00-04:00  204.8000
2025-10-30 09:31:00-04:00  204.8917
2025-10-30 09:32:00-04:00  203.9900
2025-10-30 09:33:00-04:00  203.4143
2025-10-30 09:34:00-04:00  202.9141


In [16]:
# What bars are feeding the HMA at 09:35?
ts = '2025-10-30 09:35:00-04:00'
idx = df.index.get_loc(ts)

# Last 20 closes feeding HMA
last_20 = df.iloc[idx-19:idx+1][['close']]
print("Last 20 bars feeding HMA at 09:35:")
print(last_20)

Last 20 bars feeding HMA at 09:35:
                              close
timestamp                          
2025-10-29 15:47:00-04:00  206.7600
2025-10-29 15:48:00-04:00  206.8800
2025-10-29 15:49:00-04:00  206.9100
2025-10-29 15:50:00-04:00  207.4500
2025-10-29 15:51:00-04:00  207.3500
2025-10-29 15:52:00-04:00  207.1300
2025-10-29 15:53:00-04:00  206.7850
2025-10-29 15:54:00-04:00  206.7400
2025-10-29 15:55:00-04:00  206.8800
2025-10-29 15:56:00-04:00  206.9100
2025-10-29 15:57:00-04:00  206.8200
2025-10-29 15:58:00-04:00  207.0050
2025-10-29 15:59:00-04:00  207.0500
2025-10-29 16:00:00-04:00  207.3300
2025-10-30 09:30:00-04:00  204.8000
2025-10-30 09:31:00-04:00  204.8917
2025-10-30 09:32:00-04:00  203.9900
2025-10-30 09:33:00-04:00  203.4143
2025-10-30 09:34:00-04:00  202.9141
2025-10-30 09:35:00-04:00  202.3550


In [17]:
# Recalculate SSL without momentum filter
df['ssl_score_bear_no_filter'] = df['ssl_bear_confirm'].astype(int)

ts = '2025-10-30 09:35:00-04:00'
print(f"SSL Score (with filter): {df.loc[ts, 'ssl_score_bear']}")
print(f"SSL Score (no filter):   {df.loc[ts, 'ssl_score_bear_no_filter']}")

# What would total be?
total = 1 + df.loc[ts, 'ssl_score_bear_no_filter'] + df.loc[ts, 'wae_score_bear'] + df.loc[ts, 'qqe_score_bear'] + df.loc[ts, 'vol_score']
print(f"\nTotal with no SSL filter: {total}/5")

SSL Score (with filter): 0
SSL Score (no filter):   1

Total with no SSL filter: 5/5


In [18]:
# Test Oct 29 with SSL filter off
test_date = '2025-10-29'
day_df = df[df.index.date.astype(str) == test_date].copy()
orb_high = day_df['orb_high'].iloc[-1]

# Find LONG breakout
for ts, row in day_df.iterrows():
    if not row['orb_complete']:
        continue
    if row['close'] > orb_high:
        ssl_no_filter = 1 if row['ssl_bull_confirm'] else 0
        total = 1 + ssl_no_filter + row['wae_score_bull'] + row['qqe_score_bull'] + row['vol_score']
        
        print(f"Oct 29 LONG at {ts.strftime('%H:%M')}")
        print(f"SSL (no filter): {ssl_no_filter}")
        print(f"WAE: {row['wae_score_bull']}")
        print(f"QQE: {row['qqe_score_bull']}")
        print(f"Vol: {row['vol_score']}")
        print(f"Total: {total}/5")
        print(f"\nPine showed: 5/5 (or 4/5 without volume)")
        break

Oct 29 LONG at 09:36
SSL (no filter): 1
WAE: 0
QQE: 1
Vol: 0
Total: 3/5

Pine showed: 5/5 (or 4/5 without volume)


In [19]:
# Debug WAE for Oct 29 at breakout
test_date = '2025-10-29'
day_df = df[df.index.date.astype(str) == test_date].copy()
orb_high = day_df['orb_high'].iloc[-1]

for ts, row in day_df.iterrows():
    if not row['orb_complete']:
        continue
    if row['close'] > orb_high:
        print(f"=== WAE Debug Oct 29 at {ts.strftime('%H:%M')} ===")
        print(f"Trend Up: {row['wae_trend_up']:.2f}")
        print(f"Explosion Line: {row['wae_explosion']:.2f}")
        print(f"Bull Confirm (trend > explosion): {row['wae_bull_confirm']}")
        
        # Get previous bar
        idx = df.index.get_loc(ts)
        prev = df.iloc[idx-1]
        print(f"\nTrend Up Prev: {prev['wae_trend_up']:.2f}")
        print(f"Accel Bull (trend growing + confirm): {row['wae_accel_bull']}")
        print(f"WAE Score: {row['wae_score_bull']}")
        
        print(f"\n=== Volume Debug ===")
        print(f"Volume: {row['volume']:,.0f}")
        print(f"Vol Avg: {row['vol_avg']:,.0f}")
        print(f"Vol Ratio: {row['vol_ratio']:.2f}")
        print(f"Vol Score Raw: {row['vol_score_raw']}")
        break

=== WAE Debug Oct 29 at 09:36 ===
Trend Up: 35.55
Explosion Line: 17.35
Bull Confirm (trend > explosion): True

Trend Up Prev: 39.25
Accel Bull (trend growing + confirm): False
WAE Score: 0

=== Volume Debug ===
Volume: 3,206,437
Vol Avg: 2,302,806
Vol Ratio: 1.39
Vol Score Raw: 0.5


In [20]:
# Pine-style rounding (0.5 rounds UP to 1, not down to 0)
import math

# Test the difference
py_round = round(0.5)
pine_round = math.floor(0.5 + 0.5)  # or int(0.5 + 0.5)

print(f"Python round(0.5): {py_round}")
print(f"Pine math.round(0.5): {pine_round}")

# Fix volume score
df['vol_score_pine'] = df['vol_score_raw'].apply(lambda x: int(x + 0.5))

ts = '2025-10-29 09:36:00-04:00'
print(f"\nOct 29 Vol Score (Python rounding): {df.loc[ts, 'vol_score']}")
print(f"Oct 29 Vol Score (Pine rounding):   {df.loc[ts, 'vol_score_pine']}")

Python round(0.5): 0
Pine math.round(0.5): 1

Oct 29 Vol Score (Python rounding): 0
Oct 29 Vol Score (Pine rounding):   1


In [21]:
# Oct 29 with all fixes
ts = '2025-10-29 09:36:00-04:00'
row = df.loc[ts]

ssl = 1 if row['ssl_bull_confirm'] else 0
wae = 1 if row['wae_bull_confirm'] else 0  # No accel filter
qqe = row['qqe_score_bull']
vol = int(row['vol_score_raw'] + 0.5)  # Pine rounding

total = 1 + ssl + wae + qqe + vol
print(f"=== Oct 29 with fixes ===")
print(f"ORB: 1")
print(f"SSL: {ssl}")
print(f"WAE: {wae}")
print(f"QQE: {qqe}")
print(f"Vol: {vol}")
print(f"Total: {total}/5")
print(f"\nPine: 5/5")

=== Oct 29 with fixes ===
ORB: 1
SSL: 1
WAE: 1
QQE: 1
Vol: 1
Total: 5/5

Pine: 5/5


## Validation Checklist

Compare Python output with TradingView confluence table:

| Component | Pine | Python | Match? |
|-----------|------|--------|--------|
| SSL       |      |        |        |
| WAE       |      |        |        |
| QQE       |      |        |        |
| Volume    |      |        |        |
| **Total** |      |        |        |

If mismatches exist, use Cell 9 debug output to identify which component is wrong.